# 10.1: Train a Neural Network on MNIST

Welcome to the lab! As we just said, the plan is to build a small neural network from scratch in NumPy, train it on a handwritten-digit task, evaluate it, then prove the same job in three lines of scikit-learn.

**A note on the dataset**: we're using scikit-learn's bundled `load_digits` (1,797 examples at 8x8 pixels) rather than the full 70,000-example, 28x28 MNIST. JupyterLite runs in your browser via Pyodide and is slower than native Python; the smaller dataset gives the same lesson at a fraction of the training time. The maths generalises directly to full MNIST.

## Setup

Imports and a fixed random seed for reproducibility.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(42)
np.random.seed(42)

## Step 1: Load the data and look at it

`load_digits` returns 1,797 examples, each an 8x8 grayscale image of a handwritten digit. Pixel intensities are integers from 0 to 16. Labels are integers 0 to 9.

In [ ]:
digits = load_digits()
X_all = digits.data       # shape: (1797, 64)
y_all = digits.target     # shape: (1797,)

print('Inputs shape:', X_all.shape)
print('Labels shape:', y_all.shape)
print('Pixel range:', X_all.min(), 'to', X_all.max())
print('Class labels:', np.unique(y_all))

Now visualise a handful of examples. Look at the variation: two 4s do not always look alike, and some 7s look more like 1s. The network's job is to find what is invariant across the variation.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, image, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(image, cmap='gray_r')
    ax.set_title(int(label))
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Sample digits')
plt.tight_layout()
plt.show()

## Step 2: Prepare the data

We split into training and test sets, normalise pixel values to the range [0, 1], and one-hot encode the labels.

- **Train/test split** so we can measure performance on examples the network has not been trained on.
- **Normalising** because gradient descent works much better when features are on a similar scale.
- **One-hot encoding** because cross-entropy loss compares predicted class probabilities to a target probability distribution. The integer label `7` becomes the vector `[0,0,0,0,0,0,0,1,0,0]`.

In [ ]:
# 80/20 train/test split, stratified so each class is represented in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)

# Normalise pixels: load_digits has values in [0, 16]
X_train = X_train / 16.0
X_test = X_test / 16.0

# One-hot encode the labels
def one_hot(y, n_classes=10):
    out = np.zeros((y.shape[0], n_classes))
    out[np.arange(y.shape[0]), y] = 1.0
    return out

Y_train = one_hot(y_train)
Y_test = one_hot(y_test)

print('Train:', X_train.shape, Y_train.shape)
print('Test :', X_test.shape, Y_test.shape)

## Step 3: Initialise the network

We're building a two-layer fully-connected network:

- Input: 64 pixels.
- Hidden layer: 32 neurons with ReLU.
- Output layer: 10 neurons with softmax.

Weights are initialised with small random values (scaled by `1/sqrt(fan_in)`, the *He initialisation* trick that helps deep networks train). Biases start at zero.

The cell creates four arrays: `W1` shape `(64, 32)`, `b1` shape `(32,)`, `W2` shape `(32, 10)`, `b2` shape `(10,)`. It also prints a small slice of `W1` so you can actually see the values. Try changing the seed and re-running — the numbers will shift, but training will still converge.

In [ ]:
input_size = 64
hidden_size = 32
output_size = 10

def init_params(input_size, hidden_size, output_size, seed=0):
    rng = np.random.default_rng(seed)
    W1 = rng.standard_normal((input_size, hidden_size)) * np.sqrt(2.0 / input_size)
    b1 = np.zeros(hidden_size)
    W2 = rng.standard_normal((hidden_size, output_size)) * np.sqrt(2.0 / hidden_size)
    b2 = np.zeros(output_size)
    return W1, b1, W2, b2

W1, b1, W2, b2 = init_params(input_size, hidden_size, output_size, seed=0)
print('W1:', W1.shape, 'b1:', b1.shape, 'W2:', W2.shape, 'b2:', b2.shape)
print('\nFirst 3 rows, first 5 columns of W1:')
print(W1[:3, :5].round(3))

## Step 4: The forward pass

Three pieces:

**`relu(x)`** — outputs zero for negative inputs, the input itself otherwise. This is the hidden layer's activation: the non-linearity that lets stacking layers buy you real expressive power. Each neuron's output is transformed independently.

**`softmax(x)`** — you saw ReLU and sigmoid earlier in the module, but softmax is unlike those. ReLU and sigmoid are element-wise: they transform each neuron's output independently, with no neuron knowing what the others are doing. Softmax is different — instead of operating on a single neuron, it takes care of the activation of the entire output layer at once. It looks at all ten raw scores together and asks: given these numbers, what probability should we assign to each class? It does this by exponentiating each score and dividing by the sum of all ten, which forces the outputs to sum to 1. The result is a proper probability distribution: "3% chance this is a 0, 2% chance it's a 1, 42% chance it's an 8..." The class with the highest probability is the prediction.

This also means softmax is not there to introduce non-linearity for learning — that's ReLU's job in the hidden layer. Softmax is output normalisation: making the final layer's raw scores interpretable as probabilities. You only ever see it at the last layer.

The `- x.max(...)` line is a numerical trick: exponentials grow fast, and subtracting the row maximum before calling `np.exp` keeps the numbers from overflowing. The probabilities come out identical — it's purely to keep the arithmetic stable.

**`forward(...)`** chains both: weighted sum → ReLU → weighted sum → softmax. Two layers, two different jobs. Read the four lines slowly — each one is a direct translation of something covered in *From neuron to network*: weighted sum and bias, ReLU activation, weighted sum and bias again, softmax to produce probabilities. If that felt abstract, here it is in code.

In [ ]:
def relu(x):
    return np.maximum(0, x)

def softmax(x):
    # Subtract row max for numerical stability
    shifted = x - x.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    a2 = softmax(z2)
    return a2, z1, a1

## Step 5: Cross-entropy loss

The loss is a single number measuring how wrong the network's predictions are across the batch. The formula is `-log(predicted probability of the correct class)` — so if the network assigns 90% confidence to the right answer, the loss is `-log(0.9) ≈ 0.10`. If it assigns 1% confidence, the loss is `-log(0.01) ≈ 4.60`. The more wrong it is, the higher the number.

`eps = 1e-9` is there because `log(0)` is negative infinity. If softmax ever assigned exactly zero probability to the correct class, the loss would blow up to infinity and break gradient descent. In practice, softmax can't quite reach zero for finite inputs, but floating-point arithmetic can round a very small value down to exactly 0. Adding `eps` (one billionth) means we always compute `log(probability + 1e-9)` instead of `log(probability)` — it makes no meaningful difference when the probability is any reasonable size, but it prevents the degenerate case from crashing training.

**Sanity check.** Now let's run the next cell and look at the initial loss. We're expecting it to be around 2.3. This is because so far, all we've done is initialise an empty neural network with random weights — it has no ability to predict which digit the input is, so we'd expect it to assign roughly 10% probability to each of the ten classes. Cross-entropy calculates loss as `-log(predicted probability of the correct class)`, so with a 10% prediction that's `-log(0.1) = log(10) ≈ 2.303`. If you see a dramatically different number at this stage, something has likely gone wrong upstream — most probably in the one-hot encoding.

In [ ]:
def cross_entropy(a2, Y):
    eps = 1e-9
    return -np.sum(Y * np.log(a2 + eps)) / a2.shape[0]

a2, _, _ = forward(X_train, W1, b1, W2, b2)
print(f'Initial loss: {cross_entropy(a2, Y_train):.4f} (expect ~2.30)')

## Step 6: The backward pass

This is the cell to spend the most time on. Each line corresponds to something *How neural networks learn* was describing in the abstract.

The backward pass answers one question: which weights were most responsible for the error, and in which direction should they move to reduce it? It does this by working backwards from the output — computing the gradient of the loss with respect to each parameter, layer by layer. Each layer passes a signal back to the one before it, saying "here is how much you contributed to the mistake." That signal is the gradient.

Before reading the bullets, the variable names follow a consistent pattern: `d` means "derivative of the loss with respect to this quantity", `z` is the pre-activation value (the raw weighted sum before the activation function is applied), `a` is the post-activation value (after ReLU or softmax), `W` is weights, `b` is biases, and the number is the layer. So `dz2` is the derivative with respect to the output layer's raw scores, `da1` is the derivative with respect to the hidden layer's activated outputs, and so on.

- `dz2` is the gradient at the output layer — how wrong were the raw scores, and in which direction? Computing this properly requires applying the chain rule through softmax and then through cross-entropy. When you work through the maths, the derivatives of those two functions cancel almost entirely, leaving just `(predictions - targets) / batch_size`. This is a well-known simplification and one of the reasons softmax and cross-entropy are so commonly paired — the gradient is about as clean as it could be.
- `dW2` and `db2` are the gradients for the second layer's weights and biases — how much did each weight in that layer contribute to the error at the output?
- `da1` is the gradient flowing back into the hidden layer. The output layer is distributing blame to each hidden neuron proportionally to how much it influenced the wrong output — neurons with large weights to the wrong output classes get more blame.
- `dz1` applies the ReLU derivative, which is `1` where the neuron was active (input > 0) and `0` where it was not. Neurons that were off during the forward pass contributed nothing to the output, so they receive no gradient — there is nothing to correct in a neuron that wasn't firing.
- `dW1` and `db1` are the gradients for the first layer's parameters, computed the same way as `dW2` and `db2`.

Six lines and you have backpropagation.

In [ ]:
def backward(X, Y, a2, z1, a1, W1, W2):
    m = X.shape[0]
    dz2 = (a2 - Y) / m
    dW2 = a1.T @ dz2
    db2 = dz2.sum(axis=0)
    da1 = dz2 @ W2.T
    dz1 = da1 * (z1 > 0)
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0)
    return dW1, db1, dW2, db2

## Step 7: Train

The training loop is one cell. For each epoch we shuffle the training data, split it into mini-batches of 32 examples, and for each batch:

1. Forward pass to get predictions.
2. Compute the loss.
3. Backward pass to get gradients.
4. Update each parameter: `W1 -= learning_rate * dW1`, and so on.

We track the average loss over each epoch so we can plot it.

In [ ]:
def train(X, Y, W1, b1, W2, b2, *, epochs=30, batch_size=32, lr=0.1):
    n = X.shape[0]
    losses = []
    for epoch in range(epochs):
        # Shuffle each epoch to keep gradient estimates unbiased
        idx = np.random.permutation(n)
        Xs = X[idx]
        Ys = Y[idx]
        epoch_loss = 0.0
        n_batches = 0
        for start in range(0, n, batch_size):
            end = start + batch_size
            xb = Xs[start:end]
            yb = Ys[start:end]
            a2, z1, a1 = forward(xb, W1, b1, W2, b2)
            epoch_loss += cross_entropy(a2, yb)
            n_batches += 1
            dW1, db1, dW2, db2 = backward(xb, yb, a2, z1, a1, W1, W2)
            W1 -= lr * dW1
            b1 -= lr * db1
            W2 -= lr * dW2
            b2 -= lr * db2
        losses.append(epoch_loss / n_batches)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch + 1:>3}: loss = {losses[-1]:.4f}')
    return W1, b1, W2, b2, losses

W1, b1, W2, b2 = init_params(input_size, hidden_size, output_size, seed=0)
W1, b1, W2, b2, losses = train(X_train, Y_train, W1, b1, W2, b2,
                                epochs=30, batch_size=32, lr=0.1)

Now plot the training loss curve. It should fall steadily, perhaps with some noise from the mini-batch sampling.

**Diagnostics:** if the curve is flat, your gradient is vanishing or your learning rate is too small. If it bounces wildly or trends upward, your learning rate is too large. Try changing `lr` in the cell above to `1.0` and to `0.001` and re-running.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.title('Loss curve')
plt.grid(alpha=0.3)
plt.show()

## Step 8: Evaluate, and look at failures

Compute test accuracy. With the defaults you should land around 95%.

In [ ]:
def predict(X, W1, b1, W2, b2):
    a2, _, _ = forward(X, W1, b1, W2, b2)
    return a2.argmax(axis=1)

y_pred = predict(X_test, W1, b1, W2, b2)
test_acc = (y_pred == y_test).mean()
print(f'Test accuracy: {test_acc:.3f}')

Build a confusion matrix to show which digits get confused with which. The mistakes are not random: 4s and 9s tend to swap, 1s and 7s blur, and 5s and 8s sometimes look alike.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion matrix')
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black',
                    fontsize=9)
plt.colorbar(im)
plt.tight_layout()
plt.show()

Look at a few misclassified examples. Some are clearly the network's fault. A handful are genuinely ambiguous, even to a human reader: a 95% model is not 100% because the underlying problem is not 100% answerable from 64 chunky pixels.

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print(f'Number wrong: {len(wrong)} / {len(y_test)}')

if len(wrong) > 0:
    n_show = min(8, len(wrong))
    fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2.5))
    for ax, idx in zip(np.atleast_1d(axes), wrong[:n_show]):
        ax.imshow(X_test[idx].reshape(8, 8), cmap='gray_r')
        ax.set_title(f'true {y_test[idx]} / pred {y_pred[idx]}', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    plt.suptitle('Misclassified examples')
    plt.tight_layout()
    plt.show()

## Step 9: The same task in three lines of scikit-learn

Here is the punchline. Three lines (after the import). The accuracy should be in the same ballpark as your from-scratch network, give or take a percentage point.

One thing to note: `MLPClassifier` uses the Adam optimiser by default, whereas our hand-rolled network used plain SGD. They are both doing the same thing — forward pass, backward pass, weight updates — but Adam adapts its learning rate as it trains, which is why it can match our network's accuracy in a comparable number of iterations despite being a different optimiser. The library version is not *better*. It is just *hidden*.

In [ ]:
from sklearn.neural_network import MLPClassifier

clf = MLPClassifier(hidden_layer_sizes=(32,), max_iter=100, tol=5e-3, random_state=42)
clf.fit(X_train, y_train)
print(f'sklearn test accuracy: {clf.score(X_test, y_test):.3f}')
print(f'from-scratch test accuracy: {test_acc:.3f}')

## Hyperparameter exploration

Now that the basic version works, try a few variations. The cells below give you starting points; change the parameters and re-run.

**Bigger hidden layer.** More capacity, more weights, slightly more compute per epoch.

In [ ]:
clf_big = MLPClassifier(hidden_layer_sizes=(128,), max_iter=100, tol=5e-3, random_state=42)
clf_big.fit(X_train, y_train)
print(f'(128,) test accuracy: {clf_big.score(X_test, y_test):.3f}')

**Two hidden layers.** Greater depth, hierarchical features, longer training.

In [ ]:
clf_deep = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=100, tol=5e-3, random_state=42)
clf_deep.fit(X_train, y_train)
print(f'(64, 32) test accuracy: {clf_deep.score(X_test, y_test):.3f}')

**SGD optimiser** instead of Adam. Plain stochastic gradient descent — the same algorithm we implemented from scratch. Adam is generally faster to converge; SGD can match it but usually needs more iterations and careful tuning of the learning rate. You'll likely see a convergence warning here, which illustrates the point: with the same architecture and the same number of iterations, Adam gets there and SGD doesn't.

In [ ]:
clf_sgd = MLPClassifier(hidden_layer_sizes=(32,), max_iter=100,
                        solver='sgd', random_state=42)
clf_sgd.fit(X_train, y_train)
print(f'SGD test accuracy: {clf_sgd.score(X_test, y_test):.3f}')

## What you have done

You wrote a neural network from scratch. Forward pass, cross-entropy loss, backward pass, training loop, evaluation, all in one notebook. You then ran the same task through scikit-learn and saw the accuracy match.

The code on screen is the same code that, scaled to billions of parameters and millions of dollars of compute, produces GPT-4 and Claude. The only thing that changes is the size of the matrices and the patience of the budget.

**Extension ideas:**

- Add a third layer to the from-scratch network. You will need a `W3, b3`, an extra forward step, and an extra backward step.
- Replace ReLU with `tanh`. Its derivative is `1 - tanh(x) ** 2`. Swap it in both the forward and backward passes.
- Move to full MNIST. Replace `load_digits()` with `fetch_openml('mnist_784')` from `sklearn.datasets`. The maths does not change; only the input dimensions and dataset size grow.

Return to the course when you're done and we'll summarise what we've covered.